# Europa Colab training driver

Set Colab to **Runtime → Change runtime type → GPU** first. Then edit the variables in the next cell and run downward.

This version is tuned for the `fixed_meaning` input scheme: frozen hand-built token embeddings, fixed positional encoding, and a learned transformer/output head. It installs the repo into the active Colab kernel, then imports and calls project APIs directly so training progress streams as normal cell output. `uv` is used only for setup/dependency installation.

In [ ]:
# === EDIT THESE ===
REPO_URL = "https://github.com/alvertremantel/europa.git"
BRANCH = "dev"
PROJECT_DIR = "/content/europa"

# Keep DATA_DIR/RUN_DIR on Drive so generated data and checkpoints survive Colab resets.
USE_GOOGLE_DRIVE = True
DRIVE_ROOT = "/content/drive/MyDrive/europa-colab"
DATA_DIR = f"{DRIVE_ROOT}/data/fixed-meaning"
RUN_DIR = f"{DRIVE_ROOT}/runs/fixed-meaning-small"
CONFIG_PATH = f"{DRIVE_ROOT}/train-config.toml"

# Dataset generation. Set GENERATE_DATASET=False if DATA_DIR already has train/val/test/meta.toml.
GENERATE_DATASET = True
FORCE_REGENERATE_DATASET = False
DATASET_SEED = 42

# Runtime/resume. Resume is explicit: point RESUME_FROM at a checkpoint path such as
# f"{RUN_DIR}/checkpoint-last.pt" when continuing an interrupted run.
DEVICE = "cuda"
SEED = 42
RESUME_FROM = ""  # explicit checkpoint path, or blank to disable
ADDITIONAL_EPOCHS = None  # e.g. 2 to add two epochs when resuming

# Model shape. d_model must be divisible by n_heads.
SEQUENCE_LENGTH = 64
D_MODEL = 128
N_HEADS = 4
N_LAYERS = 4
MLP_HIDDEN = 512
DROPOUT = 0.1
POSITION_ENCODING = "fixed_meaning"

# Optimization/logging. Raise/lower BATCH_SIZE based on GPU memory.
BATCH_SIZE = 256
EPOCHS = 5
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 0.1
GRAD_CLIP = 1.0
LOG_INTERVAL = 100
MAX_NEW_TOKENS = 24

# Evaluation. Increase PROGRESS_INTERVAL for more frequent inline updates.
EVALUATION_SPLITS = ["train", "val", "test"]
EVALUATION_SAMPLE_SIZE_PER_KIND = 50
EVALUATION_SAMPLE_SEED = 42
EVALUATION_FAILURES_PER_KIND = 3
EVALUATION_PROGRESS_INTERVAL_KINDS = 10


In [ ]:
from pathlib import Path
import importlib
import os
import shutil
import subprocess
import sys

def run(cmd, cwd=None):
    cmd = [str(part) for part in cmd]
    print("\n$", " ".join(cmd), flush=True)
    subprocess.run(cmd, cwd=str(cwd) if cwd else None, check=True)

def enable_line_buffering():
    # Helps Colab/Jupyter render long-running Python output promptly.
    for stream in (sys.stdout, sys.stderr):
        if hasattr(stream, "reconfigure"):
            stream.reconfigure(line_buffering=True)

enable_line_buffering()

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    Path(DRIVE_ROOT).mkdir(parents=True, exist_ok=True)

run([sys.executable, "-m", "pip", "install", "-q", "uv"])

project = Path(PROJECT_DIR)
if not (project / ".git").exists():
    if project.exists():
        shutil.rmtree(project)
    run(["git", "clone", "--branch", BRANCH, REPO_URL, project])
else:
    run(["git", "fetch", "origin"], cwd=project)
    run(["git", "checkout", BRANCH], cwd=project)
    run(["git", "pull", "--ff-only"], cwd=project)

# Direct imports run inside this notebook's Python process, so the kernel itself must
# satisfy the package's Python requirement.
if not ((3, 12) <= sys.version_info[:2] < (3, 13)):
    raise RuntimeError(
        f"Direct in-kernel training requires Python 3.12; this kernel is {sys.version.split()[0]}. "
        "Switch to a Python 3.12 Colab runtime, or use the older uv-run subprocess flow."
    )

# Install the project into the active kernel environment, not a separate uv venv.
# Keeping it editable means a git pull is enough for code changes to be imported.
run(["uv", "pip", "install", "--system", "--upgrade", "-e", "."], cwd=project)

os.chdir(project)
if str(project) not in sys.path:
    sys.path.insert(0, str(project))
importlib.invalidate_caches()

import torch
import eur_ts
print(sys.version, flush=True)
print('eur_ts import:', Path(eur_ts.__file__).resolve(), flush=True)
print('torch', torch.__version__, 'cuda?', torch.cuda.is_available(), 'torch cuda', torch.version.cuda, flush=True)
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO CUDA DEVICE', flush=True)


In [ ]:
from dataclasses import asdict

from eur_ts.artifacts import toml_text
from eur_ts.generator.core import Config as GeneratorConfig, generate_dataset

data_path = Path(DATA_DIR)
if GENERATE_DATASET:
    if FORCE_REGENERATE_DATASET and data_path.exists():
        shutil.rmtree(data_path)
    if (data_path / "train.txt").exists() and (data_path / "meta.toml").exists():
        print(f"Dataset already exists at {data_path}; skipping generation.")
    else:
        data_path.parent.mkdir(parents=True, exist_ok=True)
        generator_config = GeneratorConfig(seed=DATASET_SEED, output_dir=str(data_path), validate=True)
        print(toml_text({"generator": asdict(generator_config)}).rstrip(), flush=True)
        generate_dataset(generator_config)
else:
    print(f"Using existing dataset at {data_path}")


In [ ]:
import json

from eur_ts.config.sizing import model_size_from_config
from eur_ts.config.toml_io import load_train_config
from eur_ts.artifacts import toml_text

assert D_MODEL % N_HEADS == 0, "D_MODEL must be divisible by N_HEADS"
assert POSITION_ENCODING == "fixed_meaning", "Unsupported position encoding"
if ADDITIONAL_EPOCHS is not None and not str(RESUME_FROM).strip():
    raise ValueError("Set RESUME_FROM to an explicit checkpoint path when using ADDITIONAL_EPOCHS")

def toml_str(value):
    return json.dumps(str(value))

def toml_optional_int(value):
    return '""' if value is None else str(int(value))

config_text = f'''# Generated by notebooks/colab-training.ipynb

[paths]
data_dir = {toml_str(DATA_DIR)}
output_dir = {toml_str(RUN_DIR)}

[runtime]
device = {toml_str(DEVICE)}
seed = {SEED}

[resume]
resume_from = {toml_str(RESUME_FROM)}
additional_epochs = {toml_optional_int(ADDITIONAL_EPOCHS)}

[model]
sequence_length = {SEQUENCE_LENGTH}
d_model = {D_MODEL}
n_heads = {N_HEADS}
n_layers = {N_LAYERS}
mlp_hidden = {MLP_HIDDEN}
dropout = {DROPOUT}
position_encoding = {toml_str(POSITION_ENCODING)}

[optimization]
batch_size = {BATCH_SIZE}
epochs = {EPOCHS}
learning_rate = {LEARNING_RATE}
weight_decay = {WEIGHT_DECAY}
grad_clip = {GRAD_CLIP}

[logging]
log_interval = {LOG_INTERVAL}
max_new_tokens = {MAX_NEW_TOKENS}

[training]
training_mode = "token_stream"
training_format = "final_only"
skip_overlong_examples = false
curriculum_name = ""
'''

config_path = Path(CONFIG_PATH)
config_path.parent.mkdir(parents=True, exist_ok=True)
config_path.write_text(config_text)
train_config = load_train_config(config_path)
print(config_text, flush=True)
print(toml_text({"model_size": model_size_from_config(train_config)}).rstrip(), flush=True)


In [ ]:
# Long-running cell. If Colab disconnects, rerun setup, then set RESUME_FROM to
# f"{RUN_DIR}/checkpoint-last.pt" (optionally with ADDITIONAL_EPOCHS) before rerunning.
# This calls the training loop directly, so progress prints are normal notebook output.
from eur_ts.config.toml_io import load_train_config
from eur_ts.trainer.core import train_model

train_config = load_train_config(Path(CONFIG_PATH))
train_model(train_config)


In [ ]:
from types import SimpleNamespace
import torch

from eur_ts.evaluator.cli import print_console_summary
from eur_ts.evaluator.metadata import (
    checkpoint_payload,
    expected_available_kind_counts,
    kind_definitions_from_metadata,
    load_metadata,
    ordered_categories,
    resolve_data_dir,
    resolve_max_new_tokens,
    resolve_output_prefix,
    skipped_kinds_from_metadata,
)
from eur_ts.evaluator.runner import run_evaluation
from eur_ts.evaluator.sampling import (
    collect_selected_examples,
    print_selection_summary,
    validate_available_counts,
)
from eur_ts.trainer.utils import configure_runtime, resolve_device

best_checkpoint = Path(RUN_DIR) / "checkpoint-best.pt"
if not best_checkpoint.exists():
    raise FileNotFoundError(f"No best checkpoint found at {best_checkpoint}")
if EVALUATION_SAMPLE_SIZE_PER_KIND <= 0:
    raise ValueError("EVALUATION_SAMPLE_SIZE_PER_KIND must be positive")
if EVALUATION_FAILURES_PER_KIND < 0:
    raise ValueError("EVALUATION_FAILURES_PER_KIND must be non-negative")
if EVALUATION_PROGRESS_INTERVAL_KINDS < 0:
    raise ValueError("EVALUATION_PROGRESS_INTERVAL_KINDS must be non-negative")

eval_args = SimpleNamespace(
    checkpoint=str(best_checkpoint),
    data_dir=str(DATA_DIR),
    splits=EVALUATION_SPLITS,
    device=DEVICE,
    max_new_tokens=None,
    sample_size_per_kind=EVALUATION_SAMPLE_SIZE_PER_KIND,
    sample_seed=EVALUATION_SAMPLE_SEED,
    output_prefix=None,
    failures_per_kind=EVALUATION_FAILURES_PER_KIND,
    progress_interval_kinds=EVALUATION_PROGRESS_INTERVAL_KINDS,
)
eval_device = resolve_device(eval_args.device)
configure_runtime(eval_device)
# Only metadata is needed here; keep checkpoint tensors off GPU until run_evaluation loads the model.
payload = checkpoint_payload(best_checkpoint, torch.device("cpu"))
eval_data_dir = resolve_data_dir(eval_args, payload)
eval_max_new_tokens = resolve_max_new_tokens(eval_args, payload)
output_prefix = resolve_output_prefix(eval_args, best_checkpoint)
metadata = load_metadata(eval_data_dir)
categories = ordered_categories(metadata)
kind_definitions = kind_definitions_from_metadata(metadata)
skipped_kinds = skipped_kinds_from_metadata(metadata)
expected_counts = expected_available_kind_counts(metadata, eval_args.splits)
selected_examples, available_counts = collect_selected_examples(
    data_dir=eval_data_dir,
    splits=eval_args.splits,
    sample_size_per_kind=eval_args.sample_size_per_kind,
    sample_seed=eval_args.sample_seed,
)
validate_available_counts(
    actual_available_counts=available_counts,
    expected_available_counts_map=expected_counts,
)
print_selection_summary(
    splits=eval_args.splits,
    sample_size_per_kind=eval_args.sample_size_per_kind,
    selected_examples=selected_examples,
    available_counts=available_counts,
)
summary = run_evaluation(
    checkpoint_path=best_checkpoint,
    data_dir=eval_data_dir,
    splits=eval_args.splits,
    device=eval_device,
    max_new_tokens=eval_max_new_tokens,
    sample_size_per_kind=eval_args.sample_size_per_kind,
    sample_seed=eval_args.sample_seed,
    output_prefix=output_prefix,
    failures_per_kind=eval_args.failures_per_kind,
    progress_interval_kinds=eval_args.progress_interval_kinds,
    metadata=metadata,
    categories=categories,
    kind_definitions=kind_definitions,
    skipped_kinds=skipped_kinds,
    expected_counts=expected_counts,
    selected_examples=selected_examples,
    available_counts=available_counts,
)
print_console_summary(summary)


In [ ]:
from eur_ts.trainer.core import load_checkpoint
from eur_ts.trainer.inference import generate_completion
from eur_ts.trainer.utils import configure_runtime, resolve_device

PROMPT = "<do> <calc> 03000000 + 03000000 ="
predict_device = resolve_device(DEVICE)
configure_runtime(predict_device)
model, tokenizer = load_checkpoint(best_checkpoint, predict_device)
prediction = generate_completion(
    model=model,
    tokenizer=tokenizer,
    prompt=PROMPT,
    max_new_tokens=MAX_NEW_TOKENS,
    device=predict_device,
)
print(prediction)
